## 环境配置
### 安装配置 R 语言环境

```
## 进入 R 编程环境
conda activate LabelTransfer

## 安装所需库
conda create -n LabelTransfer -c conda-forge \
r-base=4.3 \
hdf5 \
r-hdf5r \
r-seurat \
r-seuratobject \
r-arrow \
r-tidyverse \
r-ggplot2 \
r-ggpmisc \
r-cowplot \
r-gridextra \
r-viridis \
r-hrbrthemes \
r-jsonlite \
r-remotes
```

### 安装BPcells

```
## 下载 BPcells 包
cd ~
git clone https://github.com/bnprks/BPCells.git

## 进入 R 环境
R

## 使用本地编译安装
remotes::install_local("~/BPCells/r")
```

### 安装 ipynb 文件运行 R 所需库

```
## 安装所需库
conda install -c conda-forge \
r-irkernel \
jupyter \
notebook \
ipykernel

## 进入 R 注册 kernel
IRkernel::installspec(name = "LabelTransfer", displayname = "R_LabelTransfer")

##  数据准备

### 在终端中执行，进入你的工作目录

```
cd ./data/
wget https://s3-us-west-2.amazonaws.com/10x.files/samples/xenium/3.0.0/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs.zip
unzip Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs.zip
```

```
curl -O https://cf.10xgenomics.com/samples/cell-exp/8.0.1/17k_Ovarian_Cancer_scFFPE/17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5
```

```
curl -O https://cf.10xgenomics.com/supp/cell-exp/FLEX_Ovarian_Barcode_Cluster_Annotation.csv
```

In [1]:
# ============================================
# Cell 1: 统一初始化 - 集中管理所有配置
# ============================================

# 1. 设置工作目录
setwd("/home/ailab/caohao/AdaDiss/")

# 2. 定义所有文件路径 - 层次化文件夹结构
paths <- list(
    # ===== 项目根目录 =====
    project_root = "./",
    
    # ===== 原始数据（只读，永不修改）=====
    raw_data = list(
        flex = list(
            h5 = "./data/raw/flex/17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5",
            annotation = "./data/raw/flex/FLEX_Ovarian_Barcode_Cluster_Annotation.csv"
        ),
        xenium = list(
            dir = "./data/raw/xenium/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/",
            gene_panel = "./data/raw/xenium/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/gene_panel.json"
        )
    ),
    
    # ===== 预处理缓存（可重新生成）=====
    cache = list(
        flex = "./data/cache/ovarian_flex_data_processed.rds",
        xenium = "./data/cache/ovarian_xenium_data_processed.rds"
    ),
    
    # ===== BPCells 临时存储（大文件，可删除）=====
    bpcells = list(
        flex = "./data/bpcells/ovarian_flex_counts/",
        xenium = "./data/bpcells/ovarian_xenium_counts/"
    ),
    
    # ===== 预测结果 =====
    predictions = list(
        # 主要结果文件
        main = "./results/predictions/cell_groups.csv",
        full = "./results/predictions/cell_predictions_full.csv",
        
        # 导出给其他工具的数据
        for_gnn = "./results/exports/xenium_for_gnn.csv",
        
        # 完整Seurat对象
        seurat_objects = list(
            xenium_annotated = "./results/seurat/xenium_annotated_final.rds",
            flex_reference = "./results/seurat/flex_reference.rds"  # 可选
        )
    ),

    backup = list(
        coordinates = "./data/back/coordinates_backup.csv"
    ),
    
    # ===== Seurat 标签转移导出（供 Python 训练 notebook 对比）=====
    seurat_lt_csv = "./results/predictions/seurat_label_transfer_ovarian.csv",
    
    # ===== 分析报告 =====
    reports = list(
        stats = "./reports/prediction_stats.txt",
        logs = "./reports/logs/"
    ),
    
    # ===== 可视化 =====
    plots = list(
        dir = "./plots/",
        subdirs = list(
            quality_control = "./plots/01_quality_control/",
            annotation = "./plots/02_annotation/",
            validation = "./plots/03_validation/",
            spatial = "./plots/04_spatial/"
        )
    )
)

# 3. 辅助函数：一键创建所有目录
create_all_dirs <- function(paths) {
    dirs_to_create <- c(
        # 数据目录
        "./data/raw/flex/",
        "./data/raw/xenium/",
        "./data/cache/",        
        "./data/backup/",
        "./data/bpcells/ovarian_flex_counts/",
        "./data/bpcells/ovarian_xenium_counts/",

        
        # 结果目录
        "./results/predictions/",
        "./results/exports/",
        "./results/seurat/",
        
        # 报告目录
        "./reports/logs/",
        
        # 绘图目录
        "./plots/01_quality_control/",
        "./plots/02_annotation/",
        "./plots/03_validation/",
        "./plots/04_spatial/"
    )
    
    for(dir_path in dirs_to_create) {
        if(!dir.exists(dir_path)) {
            dir.create(dir_path, recursive = TRUE)
            cat("📁 创建目录:", dir_path, "\n")
        }
    }
}

# 4. 创建所有目录
cat("📁 检查并创建所需目录...\n")
create_all_dirs(paths)

# 5. 设置分析参数
params <- list(
    # Flex QC 参数
    flex_min_counts = 200,
    flex_max_counts = 10000,
    flex_max_mt = 10,
    
    # Xenium 参数
    xenium_npcs = 50,
    xenium_dims = 1:30,
    xenium_resolution = 0.6,
    xenium_cluster_name = "clusters",
    
    # 标签转移参数
    transfer_dims = 1:30,
    transfer_k = 50,
    
    # 可视化参数
    seed = 42,
    
    # 输出控制
    save_intermediate = TRUE,      # 是否保存中间结果
    verbose = TRUE                 # 是否打印详细信息
)

# ========== 关键：细胞类型映射表 ==========
# 6. 将分数列名（点号格式）映射到原始细胞类型名
cell_type_mapping <- c(
    # 分数列名格式 → 原始格式
    "Tumor.Associated.Fibroblasts" = "Tumor Associated Fibroblasts",
    "Endothelial.Cells" = "Endothelial Cells",
    "Stromal.Associated.Fibroblasts" = "Stromal Associated Fibroblasts",
    "T...NK.Cells" = "T & NK Cells",
    "Malignant.Cells.Lining.Cyst" = "Malignant Cells Lining Cyst",
    "Proliferative.Tumor.Cells" = "Proliferative Tumor Cells",
    "Tumor.Cells" = "Tumor Cells",
    "Pericytes" = "Pericytes",
    "Granulosa.Cells" = "Granulosa Cells",
    "Macrophages" = "Macrophages",
    "MT.High..Jun..Fos..Tumor.Cells" = "MT-High, Jun+/Fos+ Tumor Cells",
    "VEGFA..Tumor.Cells" = "VEGFA+ Tumor Cells",
    "Smooth.Muscle.Cells" = "Smooth Muscle Cells",
    "Inflammatory.Tumor.Cells" = "Inflammatory Tumor Cells",
    "Ciliated.Epithelial.Cells" = "Ciliated Epithelial Cells",
    "Fallopian.Tube.Epithelium" = "Fallopian Tube Epithelium"
)

# 反向映射表（原始格式 → 分数列名格式）
reverse_mapping <- setNames(names(cell_type_mapping), cell_type_mapping)

# 打印映射表验证
cat("\n📋 细胞类型映射表:\n")
cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
for (i in 1:length(cell_type_mapping)) {
    cat(sprintf("  %-35s → %-40s\n", 
                names(cell_type_mapping)[i], 
                cell_type_mapping[i]))
}
cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")

# 7. 设置随机种子
set.seed(params$seed)

# 8. 加载所有必要的包
cat("📦 加载R包...\n")
suppressPackageStartupMessages({
    # 核心包
    library(Seurat)
    library(BPCells)
    library(SeuratObject)
    library(SeuratDisk)
    library(tidyverse)
    library(jsonlite)
    
    # 绘图包
    library(ggplot2)
    library(ggpmisc)
    library(scales)
    library(cowplot)
    library(gridExtra)
    library(viridis)
    library(hrbrthemes)
})

# 9. 设置Seurat内存限制
options(future.globals.maxSize = 1e9)

# 10. 定义缓存加载函数
load_or_process <- function(cache_file, process_func, force_reprocess = FALSE) {
    if (!force_reprocess && file.exists(cache_file)) {
        cat("📦 从缓存加载:", basename(cache_file), "\n")
        return(readRDS(cache_file))
    } else {
        if (force_reprocess) cat("🔄 强制重新处理...\n")
        else cat("🔄 首次运行，正在处理...\n")
        obj <- process_func()
        # 确保缓存目录存在
        cache_dir <- dirname(cache_file)
        if (!dir.exists(cache_dir)) {
            dir.create(cache_dir, recursive = TRUE)
        }
        saveRDS(obj, cache_file)
        cat("✅ 已保存到:", basename(cache_file), "\n")
        return(obj)
    }
}

# 11. 打印配置信息
cat("\n✅ 环境初始化完成\n")
cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n")
cat("📁 工作目录:", getwd(), "\n")
cat("\n📂 输入数据路径:\n")
cat("  - Flex数据:", paths$raw_data$flex$h5, "\n")
cat("  - Flex注释:", paths$raw_data$flex$annotation, "\n")
cat("  - Xenium数据:", paths$raw_data$xenium$dir, "\n")
cat("\n📤 输出数据路径:\n")
cat("  - 预测结果:", paths$predictions$main, "\n")
cat("  - 完整预测:", paths$predictions$full, "\n")
cat("  - Seurat对象:", paths$predictions$seurat_objects$xenium_annotated, "\n")
cat("  - 统计报告:", paths$reports$stats, "\n")
cat("  - 图表目录:", paths$plots$dir, "\n")
cat("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n\n")

📁 检查并创建所需目录...

📋 细胞类型映射表:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Tumor.Associated.Fibroblasts        → Tumor Associated Fibroblasts            
  Endothelial.Cells                   → Endothelial Cells                       
  Stromal.Associated.Fibroblasts      → Stromal Associated Fibroblasts          
  T...NK.Cells                        → T & NK Cells                            
  Malignant.Cells.Lining.Cyst         → Malignant Cells Lining Cyst             
  Proliferative.Tumor.Cells           → Proliferative Tumor Cells               
  Tumor.Cells                         → Tumor Cells                             
  Pericytes                           → Pericytes                               
  Granulosa.Cells                     → Granulosa Cells                         
  Macrophages                         → Macrophages                             
  MT.High..Jun..Fos..Tumor.Cells      → MT-High, Jun+/Fos+ Tumor Cells          
  VEGFA..Tumor.Cells         

Warning message:
“package ‘BPCells’ was built under R version 4.5.2”
Warning message:
“package ‘SeuratDisk’ was built under R version 4.5.2”



✅ 环境初始化完成
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📁 工作目录: /home/ailab/caohao/AdaDiss 

📂 输入数据路径:
  - Flex数据: ./data/raw/flex/17k_Ovarian_Cancer_scFFPE_count_filtered_feature_bc_matrix.h5 
  - Flex注释: ./data/raw/flex/FLEX_Ovarian_Barcode_Cluster_Annotation.csv 
  - Xenium数据: ./data/raw/xenium/Xenium_V1_Human_Ovarian_Cancer_Addon_FFPE_outs/ 

📤 输出数据路径:
  - 预测结果: ./results/predictions/cell_groups.csv 
  - 完整预测: ./results/predictions/cell_predictions_full.csv 
  - Seurat对象: ./results/seurat/xenium_annotated_final.rds 
  - 统计报告: ./reports/prediction_stats.txt 
  - 图表目录: ./plots/ 
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━



In [2]:
# ============================================
# Cell 2: 加载Flex参考数据
# ============================================

process_flex_data <- function() {
    cat("  步骤1: 加载原始数据...\n")
    flex_obj <- Read10X_h5(paths$raw_data$flex$h5) %>% CreateSeuratObject()
    
    cat("  步骤2: 配置BPCells磁盘存储...\n")
    if (!dir.exists(paths$bpcells$flex)) {
        write_matrix_dir(mat = flex_obj[["RNA"]]$counts, dir = paths$bpcells$flex)
    }
    flex_obj[['RNA']]$counts <- open_matrix_dir(dir = paths$bpcells$flex)
    
    cat("  步骤3: 计算线粒体比例...\n")
    flex_obj[["percent.mt"]] <- PercentageFeatureSet(flex_obj, pattern = "^MT-")
    
    cat("  步骤4: QC过滤...\n")
    n_before <- ncol(flex_obj)
    flex_obj <- subset(flex_obj, 
                       subset = nCount_RNA > params$flex_min_counts & 
                                nCount_RNA < params$flex_max_counts & 
                                percent.mt < params$flex_max_mt)
    cat("    过滤前:", n_before, "细胞 → 过滤后:", ncol(flex_obj), "细胞\n")
    
    cat("  步骤5: 添加细胞类型注释...\n")
    if (file.exists(paths$raw_data$flex$annotation)) {
        ann_df <- read.csv(paths$raw_data$flex$annotation)
        cell_types <- setNames(ann_df$Cell.Annotation, ann_df$Barcode)
        # 只保留当前数据中存在的细胞
        cell_types <- cell_types[names(cell_types) %in% colnames(flex_obj)]
        flex_obj <- AddMetaData(flex_obj, cell_types[colnames(flex_obj)], "cell_type")
        cat("    注释了", length(unique(flex_obj$cell_type)), "种细胞类型\n")
    } else {
        warning("找不到注释文件: ", paths$raw_data$flex$annotation)
    }
    
    return(flex_obj)
}

# 加载或处理Flex数据
flex_data.obj <- load_or_process(paths$cache$flex, process_flex_data)

cat("\n📊 Flex数据摘要:\n")
cat("  - 细胞数量:", ncol(flex_data.obj), "\n")
cat("  - 基因数量:", nrow(flex_data.obj), "\n")
cat("  - 细胞类型:", length(unique(flex_data.obj$cell_type)), "\n")

📦 从缓存加载: ovarian_flex_data_processed.rds 

📊 Flex数据摘要:
  - 细胞数量: 17050 
  - 基因数量: 18082 
  - 细胞类型: 16 


In [3]:
# ============================================
# Cell 4: 基因表达相关性分析
# ============================================

if (file.exists(paths$raw_data$xenium$gene_panel)) {
    cat("📈 计算基因表达相关性...\n")
    
    # 定义函数
    get_gex_means <- function(xenium_obj, flex_obj) {
        xen_means <- data.frame(
            mean_counts = rowMeans(xenium_obj[["Xenium"]]$counts),
            gene = rownames(xenium_obj[["Xenium"]]$counts)
        ) %>% arrange(desc(mean_counts)) %>% mutate(Rank = 1:n())
        
        flex_means <- data.frame(
            mean_counts = rowMeans(flex_obj[["RNA"]]$counts),
            gene = rownames(flex_obj[["RNA"]]$counts)
        ) %>% arrange(desc(mean_counts)) %>% mutate(Rank = 1:n())
        
        return(merge(xen_means, flex_means, by.x = "gene", by.y = "gene", all.x = TRUE))
    }
    
    # 读取基因面板
    gene_panel <- fromJSON(paths$raw_data$xenium$gene_panel)
    targets <- gene_panel$payload$targets
    panel_source <- setNames(
        data.frame(cbind(targets$source$identity$name, targets$type$data$name)), 
        c("gene_panel", "gene")
    )
    
    # 合并数据
    merged_means <- get_gex_means(xenium.obj, flex_data.obj)
    merged_means <- merge(merged_means, panel_source, by.x = "gene", by.y = "gene", all.x = TRUE) %>%
                   na.omit() %>% arrange(gene_panel)
    
    # 绘图
    p_cor <- ggplot(merged_means, aes(x = mean_counts.y, y = mean_counts.x, color = gene_panel)) +
        geom_point(size = 0.5) +
        scale_colour_manual(values = c("darkcyan", "coral")) +
        stat_poly_eq() +
        scale_x_log10() + scale_y_log10() +
        xlab("SC Flex Mean Expression") + ylab("Xenium Mean Expression") +
        ggtitle("Gene Expression Correlation") +
        theme_classic() +
        geom_abline(slope = 1, intercept = 0)
    
    print(p_cor)
    cat("✅ 相关性分析完成\n")
} else {
    cat("⚠️ 跳过相关性分析（找不到gene_panel.json）\n")
}

📈 计算基因表达相关性...


ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'rowMeans': object 'xenium.obj' not found


In [ ]:
# ============================================
# Cell 5: Flex参考数据处理
# ============================================

cat("🔧 处理Flex参考数据...\n")

DefaultAssay(flex_data.obj) <- "RNA"
flex_data.obj <- NormalizeData(flex_data.obj) %>%
                 FindVariableFeatures() %>%
                 ScaleData() %>%
                 RunPCA(verbose = FALSE) %>%
                 RunUMAP(dims = 1:15, verbose = FALSE) %>%
                 FindNeighbors(dims = 1:15) %>%
                 FindClusters(resolution = 0.5)

# 可视化
p1 <- DimPlot(flex_data.obj, reduction = "umap", group.by = "RNA_snn_res.0.5", 
              label = TRUE, pt.size = 0.2) + ggtitle("Flex Clusters")
p2 <- DimPlot(flex_data.obj, reduction = "umap", group.by = "cell_type", 
              label = TRUE, pt.size = 0.2) + ggtitle("Flex Cell Types")

# 显示
print(p1 + p2)

cat("✅ Flex数据处理完成\n")

In [ ]:
# ============================================
# Cell 6: Xenium数据降维和聚类
# ============================================

cat("🔧 处理Xenium数据（细胞数:", ncol(xenium.obj), "）\n")

DefaultAssay(xenium.obj) <- "Xenium"

xenium.obj <- NormalizeData(xenium.obj) %>%
              FindVariableFeatures() %>%
              ScaleData() %>%
              RunPCA(npcs = params$xenium_npcs, verbose = FALSE) %>%
              RunUMAP(dims = params$xenium_dims, verbose = FALSE) %>%
              FindNeighbors(reduction = "pca", dims = params$xenium_dims) %>%
              FindClusters(resolution = params$xenium_resolution, 
                          cluster.name = params$xenium_cluster_name)

# 可视化
p3 <- DimPlot(xenium.obj, group.by = params$xenium_cluster_name, label = TRUE) + 
      ggtitle("Xenium Clusters")
print(p3)

p4 <- ImageDimPlot(xenium.obj, fov = "fov", group.by = params$xenium_cluster_name, 
                   size = 0.5, dark.background = FALSE) +
      ggtitle("Xenium Spatial Clusters")
print(p4)

# 统计
cat("\n聚类统计:\n")
print(table(xenium.obj$clusters))

cat("✅ Xenium数据处理完成\n")

In [ ]:
# ============================================
# Cell 7: 标签转移
# ============================================

# 检查是否已经完成标签转移
if (!"predicted.id" %in% colnames(xenium.obj@meta.data)) {
    
    cat("🏷️  开始优化标签转移...\n")
    
    # 1. 获取共同基因
    flex_xen_common_genes <- intersect(rownames(xenium.obj), rownames(flex_data.obj))
    cat("  共同基因数量:", length(flex_xen_common_genes), "\n")
    
    if (length(flex_xen_common_genes) < 100) {
        warning("共同基因数量较少（<100），标签转移效果可能不佳")
    }
    
    # 2. 创建Flex子集并优化预处理
    cat("  创建Flex子集并优化预处理...\n")
    flex_subset <- CreateSeuratObject(
        counts = flex_data.obj[["RNA"]]$counts[flex_xen_common_genes,],
        meta = flex_data.obj@meta.data
    ) %>%
        NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
        FindVariableFeatures(nfeatures = 3000) %>%  # 增加高变基因数
        ScaleData() %>%
        RunPCA(npcs = 50, verbose = FALSE)  # 增加PC数量
    
    # 3. 获取高变基因交集（用于锚点查找）
    flex_highvar <- VariableFeatures(flex_subset)
    xenium_highvar <- VariableFeatures(xenium.obj)
    common_highvar <- intersect(flex_highvar, xenium_highvar)
    
    # 4. 选择特征基因（优先使用高变基因，如果太少则使用所有共同基因）
    if (length(common_highvar) >= 200) {
        features_to_use <- common_highvar[1:min(2000, length(common_highvar))]
        cat("  使用高变基因数量:", length(features_to_use), "\n")
    } else {
        features_to_use <- flex_xen_common_genes
        cat("  使用所有共同基因:", length(features_to_use), "\n")
    }
    
    # 5. 将counts转回内存格式（加速计算）
    cat("  准备数据...\n")
    flex_data.obj[["RNA"]]$counts <- as(flex_data.obj[["RNA"]]$counts, "dgCMatrix")
    xenium.obj[["Xenium"]]$counts <- as(xenium.obj[["Xenium"]]$counts, "dgCMatrix")
    
    # 6. 寻找锚点（优化参数）
    cat("  寻找锚点（这可能需要一些时间）...\n")
    anchors_from_flex <- FindTransferAnchors(
        reference = flex_subset,
        query = xenium.obj,
        features = features_to_use,
        dims = params$transfer_dims,
        reference.reduction = "pca",
        reduction = "pcaproject",
        k.filter = NA,      # 使用所有锚点
        k.score = 30        # 增加k.score以提高锚点质量
    )
    
    cat("  找到", length(anchors_from_flex@anchors), "个锚点\n")
    
    # 7. 转移标签（优化参数）
    cat("  转移标签...\n")
    label_transfer <- TransferData(
        anchorset = anchors_from_flex,
        refdata = flex_subset$cell_type,
        dims = params$transfer_dims,
        weight.reduction = "pcaproject",  # 使用PCA投影
        l2.norm = TRUE,                   # L2归一化提高稳定性
        k.weight = 50                     # 调整权重计算的k值
    )
    
    # 8. 查看返回的数据结构
    cat("  TransferData 返回的列名:", paste(colnames(label_transfer), collapse = ", "), "\n")
    
    # 9. 添加预测结果
    xenium.obj <- AddMetaData(xenium.obj, metadata = label_transfer)
    
    # 10. 后处理：添加置信度筛选
    if ("prediction.score.max" %in% colnames(xenium.obj@meta.data)) {
        confidence_threshold <- 0.6
        xenium.obj@meta.data$predicted.id_filtered <- 
            ifelse(xenium.obj@meta.data$prediction.score.max > confidence_threshold,
                   as.character(xenium.obj@meta.data$predicted.id),
                   "low_confidence")
        cat("  高置信度预测阈值:", confidence_threshold, "\n")
        cat("  高置信度细胞数:", sum(xenium.obj@meta.data$prediction.score.max > confidence_threshold, na.rm = TRUE), "\n")
    }
    
    # 11. 创建完整标签列的别名
    if ("predicted.id" %in% colnames(xenium.obj@meta.data)) {
        xenium.obj@meta.data$predicted.id_full <- xenium.obj@meta.data$predicted.id
    }
    if ("prediction.score.max" %in% colnames(xenium.obj@meta.data)) {
        xenium.obj@meta.data$predicted.id_full.score <- xenium.obj@meta.data$prediction.score.max
    }
    
    cat("✅ 标签转移完成\n")
    cat("\n预测细胞类型分布:\n")
    print(table(xenium.obj$predicted.id))
    
    # 12. 显示预测分数统计
    if ("prediction.score.max" %in% colnames(xenium.obj@meta.data)) {
        cat("\n预测分数统计:\n")
        print(summary(xenium.obj$prediction.score.max))
        
        # 显示各细胞类型的平均置信度
        cat("\n各细胞类型平均置信度:\n")
        score_summary <- xenium.obj@meta.data %>%
            group_by(predicted.id) %>%
            summarise(
                mean_score = mean(prediction.score.max, na.rm = TRUE),
                median_score = median(prediction.score.max, na.rm = TRUE),
                n_cells = n()
            ) %>%
            arrange(desc(mean_score))
        print(score_summary)
    }
    
    # 13. 可选：释放内存
    gc()
    
} else {
    cat("✅ 标签转移结果已存在，跳过\n")
    cat("当前预测相关列:", 
        paste(grep("predicted", colnames(xenium.obj@meta.data), value = TRUE), collapse = ", "), "\n")
}

In [ ]:
# ============================================
# Cell 8: 可视化标签转移结果
# ============================================

if ("predicted.id" %in% colnames(xenium.obj@meta.data)) {
    
    cat("📊 绘制预测结果...\n")
    
    # 1. UMAP可视化
    p5 <- DimPlot(xenium.obj, reduction = "umap", group.by = "predicted.id", 
                  label = TRUE, pt.size = 0.3, repel = TRUE) + 
          ggtitle("Predicted Cell Types - UMAP")
    print(p5)
    
    # 2. 空间可视化
    p6 <- tryCatch({
        ImageDimPlot(xenium.obj, fov = "fov", group.by = "predicted.id", 
                     size = 0.5, dark.background = FALSE) +
            ggtitle("Predicted Cell Types - Spatial")
    }, error = function(e) {
        cat("    空间可视化失败:", e$message, "\n")
        return(NULL)
    })
    if (!is.null(p6)) print(p6)
    
    # 3. 预测分数处理
    # 找出所有预测分数列（格式：prediction.score.xxx）
    score_cols <- grep("prediction\\.score\\.", colnames(xenium.obj@meta.data), value = TRUE)
    
    if (length(score_cols) > 0) {
        cat("\n📈 预测分数分析...\n")
        cat("  找到", length(score_cols), "个预测分数列\n")
        
        # 为每个细胞找到其预测类型的分数（最大分数）
        # 方法：对于每个细胞，从对应的预测类型列中提取分数
        get_max_score <- function(row) {
            cell_type <- row["predicted.id"]
            score_col <- paste0("prediction.score.", gsub(" ", ".", cell_type))
            if (score_col %in% names(row)) {
                return(as.numeric(row[score_col]))
            } else {
                return(NA)
            }
        }
        
        # 计算每个细胞的预测分数（其预测类型的分数）
        xenium.obj@meta.data$prediction_score <- apply(
            xenium.obj@meta.data, 1, get_max_score
        )
        
        cat("  预测分数统计:\n")
        print(summary(xenium.obj@meta.data$prediction_score))
        
        # 按细胞类型统计分数
        score_stats <- xenium.obj@meta.data %>%
            group_by(predicted.id) %>%
            summarise(
                mean_score = mean(prediction_score, na.rm = TRUE),
                median_score = median(prediction_score, na.rm = TRUE),
                min_score = min(prediction_score, na.rm = TRUE),
                max_score = max(prediction_score, na.rm = TRUE),
                n_cells = n()
            ) %>%
            arrange(desc(median_score))
        
        cat("\n📊 按细胞类型的预测分数统计:\n")
        print(score_stats)
        
        # 绘制小提琴图
        p7 <- ggplot(xenium.obj@meta.data, 
                     aes(x = predicted.id, y = prediction_score, fill = predicted.id)) +
            geom_violin(scale = "width", trim = TRUE) +
            stat_summary(fun = median, geom = "point", size = 0.5, color = "black") +
            scale_fill_viridis_d(alpha = 0.7) +
            theme_minimal() +
            theme(legend.position = "none",
                  axis.text.x = element_text(angle = 45, hjust = 1, size = 8),
                  plot.title = element_text(hjust = 0.5)) +
            ggtitle("Prediction Scores by Cell Type") +
            xlab("") + ylab("Prediction Score") +
            ylim(0, 1)
        print(p7)
        
        # 总体分数分布直方图
        p8 <- ggplot(xenium.obj@meta.data, aes(x = prediction_score)) +
            geom_histogram(bins = 50, fill = "steelblue", alpha = 0.7) +
            theme_minimal() +
            ggtitle("Overall Prediction Score Distribution") +
            xlab("Prediction Score") + ylab("Number of Cells") +
            xlim(0, 1)
        print(p8)
        
        # 可选：显示低质量预测的细胞（分数 < 0.5）
        low_score_cells <- sum(xenium.obj@meta.data$prediction_score < 0.5, na.rm = TRUE)
        cat(sprintf("\n⚠️ 预测分数 < 0.5 的细胞数量: %d (%.1f%%)\n", 
                    low_score_cells, low_score_cells / ncol(xenium.obj) * 100))
        
    } else {
        cat("\n⚠️ 未找到预测分数列，跳过分数分布图\n")
        cat("  当前元数据列名中包含 'score' 的:\n")
        score_cols_found <- grep("score", colnames(xenium.obj@meta.data), value = TRUE, ignore.case = TRUE)
        print(score_cols_found)
    }
    
    cat("\n✅ 可视化完成\n")
    
} else {
    cat("⚠️ 未找到预测结果，请先运行Cell 7\n")
}

In [ ]:
# ============================================
# Cell 9.1: 检查坐标信息并修复
# ============================================

cat("🔍 检查坐标信息...\n")

# 检查 Xenium 对象中的坐标可用性
cat("\n1. 检查 GetTissueCoordinates 结果:\n")
coords_check <- tryCatch({
    GetTissueCoordinates(xenium.obj, scale = NULL)
}, error = function(e) {
    cat("  错误:", e$message, "\n")
    return(NULL)
})

if (!is.null(coords_check) && nrow(coords_check) > 0) {
    cat("  ✅ GetTissueCoordinates 返回", nrow(coords_check), "个坐标\n")
    print(head(coords_check))
    
    # 关键修复：检查返回的数据结构
    cat("\n  数据结构:\n")
    cat("    列名:", paste(colnames(coords_check), collapse = ", "), "\n")
    cat("    行名示例:", paste(head(rownames(coords_check)), collapse = ", "), "\n")
    
    # 如果存在 'cell' 列，使用它作为细胞ID
    if ("cell" %in% colnames(coords_check)) {
        cat("\n  ✅ 找到 'cell' 列，包含正确的细胞ID\n")
        coords_df <- data.frame(
            cell_id = coords_check$cell,  # 使用 cell 列，而不是 rownames
            x_centroid = coords_check$x,
            y_centroid = coords_check$y,
            stringsAsFactors = FALSE
        )
        cat("  使用 'cell' 列作为细胞ID\n")
    } else {
        # 如果没有 'cell' 列，尝试使用 rownames
        cat("\n  ⚠️ 未找到 'cell' 列，使用 rownames\n")
        coords_df <- data.frame(
            cell_id = rownames(coords_check),
            x_centroid = coords_check$x,
            y_centroid = coords_check$y,
            stringsAsFactors = FALSE
        )
    }
    
    cat("  ✅ 方法1成功: GetTissueCoordinates\n")
    cat("  坐标数据框维度:", dim(coords_df), "\n")
    
} else {
    cat("  ⚠️ GetTissueCoordinates 未返回有效坐标\n")
    stop("❌ 无法获取坐标信息")
}

# 验证坐标
cat("\n2. 验证坐标数据:\n")
cat("  坐标数量:", nrow(coords_df), "\n")
cat("  坐标预览:\n")
print(head(coords_df))
cat("\n  坐标缺失情况:\n")
cat("    x_centroid 缺失:", sum(is.na(coords_df$x_centroid)), "\n")
cat("    y_centroid 缺失:", sum(is.na(coords_df$y_centroid)), "\n")

# 检查元数据中的细胞ID格式
cat("\n3. 检查元数据细胞ID:\n")
meta_cell_ids <- rownames(xenium.obj@meta.data)
cat("  元数据细胞数:", length(meta_cell_ids), "\n")
cat("  元数据细胞ID示例:", paste(head(meta_cell_ids), collapse = ", "), "\n")

# 检查匹配情况
coord_cell_ids <- coords_df$cell_id
cat("\n4. 检查ID匹配:\n")
cat("  坐标细胞ID示例:", paste(head(coord_cell_ids), collapse = ", "), "\n")

common_ids <- intersect(meta_cell_ids, coord_cell_ids)
cat("  共同细胞数:", length(common_ids), "\n")

if (length(common_ids) == 0) {
    cat("\n  ⚠️ 没有直接匹配！检查格式差异:\n")
    
    # 尝试去除可能的后缀
    meta_simple <- gsub("-.*$", "", meta_cell_ids)
    coord_simple <- gsub("-.*$", "", coord_cell_ids)
    
    cat("    简化后的元数据示例:", head(meta_simple), "\n")
    cat("    简化后的坐标示例:", head(coord_simple), "\n")
    
    common_simple <- intersect(meta_simple, coord_simple)
    cat("    简化后共同细胞数:", length(common_simple), "\n")
}

# 关键修复：将坐标添加到元数据
cat("\n5. 将坐标添加到元数据...\n")

# 创建坐标向量，按照元数据的顺序
# 使用 match 函数进行精确匹配
x_coord <- coords_df$x_centroid[match(meta_cell_ids, coords_df$cell_id)]
y_coord <- coords_df$y_centroid[match(meta_cell_ids, coords_df$cell_id)]

# 添加到元数据
xenium.obj@meta.data$x_centroid <- x_coord
xenium.obj@meta.data$y_centroid <- y_coord

# 验证添加结果
cat("\n6. 验证添加结果:\n")
valid_x <- sum(!is.na(xenium.obj@meta.data$x_centroid))
valid_y <- sum(!is.na(xenium.obj@meta.data$y_centroid))
cat("  有效 x_centroid:", valid_x, "/", ncol(xenium.obj), "\n")
cat("  有效 y_centroid:", valid_y, "/", ncol(xenium.obj), "\n")

if (valid_x > 0 && valid_y > 0) {
    cat("  ✅ 坐标已成功添加到元数据\n")
    cat("  示例坐标:\n")
    print(head(xenium.obj@meta.data[, c("x_centroid", "y_centroid")]))
} else {
    cat("  ❌ 坐标添加后仍然全为 NA\n")
    cat("\n  调试信息:\n")
    cat("    元数据细胞ID示例:", paste(head(meta_cell_ids), collapse = ", "), "\n")
    cat("    坐标细胞ID示例:", paste(head(coord_cell_ids), collapse = ", "), "\n")
    cat("    匹配结果示例:", paste(head(x_coord), collapse = ", "), "\n")
}

# 保存坐标到单独文件作为备份
coords_backup <- data.frame(
    cell_id = meta_cell_ids,
    x_centroid = xenium.obj@meta.data$x_centroid,
    y_centroid = xenium.obj@meta.data$y_centroid,
    stringsAsFactors = FALSE
)
write.csv(coords_backup, paths$backup$coordinates, row.names = FALSE)
cat("\n✅ 坐标已备份到:", paths$backup$coordinates, "\n")

cat("\n✅ 坐标修复完成\n")

In [ ]:
# ============================================
# Cell 9.2: 准备导出数据的基础结构
# ============================================

cat("📦 准备导出数据的基础结构...\n")

# 检查预测结果是否存在
if (!"predicted.id" %in% colnames(xenium.obj@meta.data)) {
    stop("⚠️ 未找到预测结果，请先运行 Cell 7 完成标签转移")
}

# 1. 准备元数据（直接从元数据中提取，不需要重新合并）
cat("\n1. 准备元数据...\n")

# 直接从元数据构建导出数据框
export_df <- xenium.obj@meta.data %>%
    rownames_to_column(var = "cell_id")

# 确保 predicted_cell_type 列存在
if (!"predicted_cell_type" %in% colnames(export_df) && "predicted.id" %in% colnames(export_df)) {
    export_df$predicted_cell_type <- export_df$predicted.id
    cat("  ✅ 已添加 predicted_cell_type 列\n")
}

# 2. 验证坐标
cat("\n2. 验证坐标数据:\n")
if ("x_centroid" %in% colnames(export_df) && "y_centroid" %in% colnames(export_df)) {
    valid_coords <- sum(!is.na(export_df$x_centroid) & !is.na(export_df$y_centroid))
    cat("  有效坐标数:", valid_coords, "/", nrow(export_df), 
        "(", round(valid_coords/nrow(export_df)*100, 1), "%)\n", sep = "")
    
    if (valid_coords == 0) {
        cat("\n  ⚠️ 警告: 坐标全为 NA，尝试从备份加载...\n")
        if (file.exists("./data/coordinates_backup.csv")) {
            backup <- read.csv("./data/coordinates_backup.csv", stringsAsFactors = FALSE)
            export_df$x_centroid <- backup$x_centroid[match(export_df$cell_id, backup$cell_id)]
            export_df$y_centroid <- backup$y_centroid[match(export_df$cell_id, backup$cell_id)]
            valid_coords <- sum(!is.na(export_df$x_centroid) & !is.na(export_df$y_centroid))
            cat("  从备份加载后，有效坐标数:", valid_coords, "\n")
        }
    }
} else {
    cat("  ⚠️ 元数据中没有坐标列\n")
}

# 3. 准备 PCA 嵌入
cat("\n3. 准备 PCA 嵌入...\n")
if ("pca" %in% Reductions(xenium.obj)) {
    pca_emb <- Embeddings(xenium.obj, reduction = "pca") %>% 
        as.data.frame() %>% 
        rownames_to_column("cell_id")
    cat("  ✅ PCA 维度:", ncol(pca_emb) - 1, "\n")
    
    # 合并 PCA 数据
    export_df <- export_df %>%
        left_join(pca_emb, by = "cell_id")
    cat("  ✅ PCA 数据已合并\n")
} else {
    stop("❌ 未找到 PCA 降维结果")
}

# 4. 添加聚类列
cat("\n4. 添加聚类列...\n")
if ("clusters" %in% colnames(export_df)) {
    export_df$cluster <- export_df$clusters
    cat("  ✅ 使用 'clusters' 作为聚类列\n")
} else if (params$xenium_cluster_name %in% colnames(export_df)) {
    export_df$cluster <- export_df[[params$xenium_cluster_name]]
    cat("  ✅ 已添加聚类列:", params$xenium_cluster_name, "\n")
} else {
    cat("  ⚠️ 未找到聚类列\n")
}

# 5. 最终验证
cat("\n5. 最终数据验证:\n")
cat("  总细胞数:", nrow(export_df), "\n")
cat("  总列数:", ncol(export_df), "\n")

key_cols <- c("cell_id", "x_centroid", "y_centroid", "predicted_cell_type")
key_cols_exist <- key_cols[key_cols %in% colnames(export_df)]
cat("  关键列存在:", paste(key_cols_exist, collapse = ", "), "\n")

if ("x_centroid" %in% colnames(export_df) && "y_centroid" %in% colnames(export_df)) {
    valid_coords <- sum(!is.na(export_df$x_centroid) & !is.na(export_df$y_centroid))
    cat("  有效坐标细胞数:", valid_coords, "/", nrow(export_df), 
        "(", round(valid_coords/nrow(export_df)*100, 1), "%)\n", sep = "")
}

cat("\n✅ 基础数据准备完成\n")

In [ ]:
# ============================================
# Cell 9.3: 计算预测分数和一致性
# ============================================

cat("📊 计算预测分数和一致性...\n")

# 查找预测分数列
score_cols <- grep("prediction\\.score\\.", colnames(export_df), value = TRUE)

if (length(score_cols) == 0) {
    cat("⚠️ 未找到预测分数列，跳过分数计算\n")
    export_df$prediction_score <- NA
    export_df$max_score_type <- NA
    export_df$prediction_consistent <- NA
} else {
    cat("找到", length(score_cols), "个预测分数列\n")
    
    # 定义映射表（从 Cell 1 继承）
    if (!exists("cell_type_mapping") || !exists("reverse_mapping")) {
        cat("  警告: 映射表不存在，使用默认映射\n")
        cell_type_mapping <- setNames(
            gsub("\\.", " ", gsub("\\.\\.", ", ", score_cols)),
            score_cols
        )
        reverse_mapping <- setNames(names(cell_type_mapping), cell_type_mapping)
    }
    
    # 计算预测分数
    cat("\n  计算预测分数...\n")
    get_prediction_score_mapped <- function(row) {
        cell_type <- row["predicted_cell_type"]
        if (!is.na(cell_type) && cell_type != "") {
            if (cell_type %in% names(reverse_mapping)) {
                score_col <- paste0("prediction.score.", reverse_mapping[cell_type])
                if (score_col %in% names(row)) {
                    return(as.numeric(row[score_col]))
                }
            }
        }
        return(NA_real_)
    }
    
    export_df$prediction_score <- apply(export_df, 1, get_prediction_score_mapped)
    
    # 计算最高分类型
    cat("\n  计算最高分类型...\n")
    get_max_score_type_mapped <- function(row) {
        scores <- as.numeric(row[score_cols])
        if (all(is.na(scores))) return(NA)
        max_idx <- which.max(scores)
        if (length(max_idx) > 0) {
            max_type_key <- gsub("prediction.score.", "", score_cols[max_idx])
            if (max_type_key %in% names(cell_type_mapping)) {
                return(cell_type_mapping[max_type_key])
            } else {
                # 后备方案
                converted <- max_type_key %>%
                    gsub("\\.\\.", ", ", .) %>%
                    gsub("\\.", " ", .)
                return(converted)
            }
        }
        return(NA)
    }
    
    export_df$max_score_type <- apply(export_df, 1, get_max_score_type_mapped)
    
    # 一致性检查
    cat("\n  检查预测一致性...\n")
    export_df$prediction_consistent <- export_df$predicted_cell_type == export_df$max_score_type
    
    consistent_count <- sum(export_df$prediction_consistent, na.rm = TRUE)
    cat("  预测一致性: ", consistent_count, "/", nrow(export_df), 
        " (", round(consistent_count / nrow(export_df) * 100, 1), "%)\n", sep = "")
    
    # 分数统计
    valid_scores <- export_df$prediction_score[!is.na(export_df$prediction_score)]
    cat("\n  预测分数统计:\n")
    print(summary(valid_scores))
}

cat("\n✅ 预测分数计算完成\n")

In [ ]:
# ============================================
# Cell 9.4: 导出基础预测结果
# ============================================

cat("📤 导出基础预测结果...\n")

# 基础结果列
base_cols <- c("cell_id", "cluster", "predicted_cell_type", 
               "prediction_score", "max_score_type", "prediction_consistent")
base_cols <- base_cols[base_cols %in% colnames(export_df)]

# 创建基础结果数据框
cell_groups <- export_df %>% select(all_of(base_cols))

# 保存
write.csv(cell_groups, paths$predictions$main, row.names = FALSE)

cat("\n✅ 基础结果已导出:", paths$predictions$main, "\n")
cat("  导出列数:", ncol(cell_groups), "\n")
cat("  导出细胞数:", nrow(cell_groups), "\n")

# 显示预览
cat("\n📊 文件预览（前10行）:\n")
print(head(cell_groups, 10))

In [ ]:
# ============================================
# Cell 9.5: 导出完整预测分数文件
# ============================================

cat("📤 导出完整预测分数文件...\n")

score_cols <- grep("prediction\\.score\\.", colnames(export_df), value = TRUE)

if (length(score_cols) > 0) {
    full_export <- export_df %>%
        select(cell_id, predicted_cell_type, prediction_score, 
               max_score_type, prediction_consistent, all_of(score_cols))
    
    write.csv(full_export, paths$predictions$full, row.names = FALSE)
    cat("✅ 完整预测分数文件已导出:", paths$predictions$full, "\n")
    cat("  文件大小:", nrow(full_export), "行,", ncol(full_export), "列\n")
} else {
    cat("⚠️ 未找到预测分数列，跳过导出\n")
}

In [ ]:
# ============================================
# Cell 9.6: 导出 GNN 专用数据
# ============================================

cat("📤 导出 GNN 专用数据...\n")

# 1. 验证坐标数据
cat("\n1. 验证坐标数据:\n")

# 检查 export_df 中的坐标
if ("x_centroid" %in% colnames(export_df) && "y_centroid" %in% colnames(export_df)) {
    valid_coords <- sum(!is.na(export_df$x_centroid) & !is.na(export_df$y_centroid))
    cat("  export_df 有效坐标数:", valid_coords, "/", nrow(export_df), "\n")
    
    if (valid_coords == 0) {
        cat("\n  ⚠️ export_df 坐标全为 NA，尝试从备份加载...\n")
        if (file.exists("./data/coordinates_backup.csv")) {
            backup <- read.csv("./data/coordinates_backup.csv", stringsAsFactors = FALSE)
            export_df$x_centroid <- backup$x_centroid[match(export_df$cell_id, backup$cell_id)]
            export_df$y_centroid <- backup$y_centroid[match(export_df$cell_id, backup$cell_id)]
            valid_coords <- sum(!is.na(export_df$x_centroid) & !is.na(export_df$y_centroid))
            cat("  从备份加载后，有效坐标数:", valid_coords, "\n")
        }
    }
} else {
    cat("  ❌ export_df 中没有坐标列\n")
    stop("无法导出 GNN 数据")
}

# 2. 选择 PCA 列
cat("\n2. 选择 PCA 特征...\n")
pca_cols <- grep("^PC_", colnames(export_df), value = TRUE)
if (length(pca_cols) == 0) {
    pca_cols <- grep("^PC", colnames(export_df), value = TRUE)
}

if (length(pca_cols) > 0) {
    if (length(pca_cols) > 50) {
        pca_cols <- pca_cols[1:50]
        cat("  限制 PCA 维度为前 50 维\n")
    }
    cat("  使用", length(pca_cols), "个 PCA 特征\n")
} else {
    cat("  ⚠️ 未找到 PCA 特征列\n")
}

# 3. 构建 GNN 数据
cat("\n3. 构建 GNN 数据...\n")

# 选择需要的列
gnn_cols <- c("cell_id", "x_centroid", "y_centroid", "predicted_cell_type", pca_cols)
gnn_cols <- gnn_cols[gnn_cols %in% colnames(export_df)]
gnn_data <- export_df %>% select(all_of(gnn_cols))

# 删除包含 NA 的行
n_before <- nrow(gnn_data)
gnn_data <- gnn_data %>% drop_na()
n_after <- nrow(gnn_data)

cat("\n  数据清洗:\n")
cat("    清洗前:", n_before, "个细胞\n")
cat("    清洗后:", n_after, "个细胞\n")
cat("    删除:", n_before - n_after, "个有缺失值的细胞\n")

if (n_after == 0) {
    cat("\n  ❌ 错误: 清洗后没有有效细胞！\n")
    cat("  调试信息:\n")
    cat("    坐标有效数:", sum(!is.na(export_df$x_centroid)), "\n")
    cat("    PCA 有效数:", sum(!is.na(export_df[[pca_cols[1]]])), "\n")
    cat("    标签有效数:", sum(!is.na(export_df$predicted_cell_type)), "\n")
} else {
    # 保存
    write.csv(gnn_data, paths$predictions$for_gnn, row.names = FALSE)
    
    cat("\n✅ GNN 专用数据已导出:", paths$predictions$for_gnn, "\n")
    cat("  包含", nrow(gnn_data), "个细胞\n")
    cat("  包含", ncol(gnn_data)-4, "个 PCA 特征\n")
    
    # 显示统计信息
    cat("\n📊 细胞类型分布 (前10):\n")
    type_counts <- table(gnn_data$predicted_cell_type)
    print(head(sort(type_counts, decreasing = TRUE), 10))
    
    # 显示坐标范围
    cat("\n📊 坐标范围:\n")
    cat("  x: [", sprintf("%.2f", min(gnn_data$x_centroid)), ", ", 
        sprintf("%.2f", max(gnn_data$x_centroid)), "]\n", sep = "")
    cat("  y: [", sprintf("%.2f", min(gnn_data$y_centroid)), ", ", 
        sprintf("%.2f", max(gnn_data$y_centroid)), "]\n", sep = "")
    
    # 显示前几行数据
    cat("\n📊 数据预览:\n")
    print(head(gnn_data[, 1:6]))
}

cat("\n✅ GNN 数据导出完成\n")

In [ ]:
# ============================================
# Cell 9.7: 保存 Seurat 对象和生成统计报告
# ============================================

cat("💾 保存 Seurat 对象...\n")

# 保存完整 Seurat 对象
saveRDS(xenium.obj, paths$predictions$seurat_objects$xenium_annotated)
cat("✅ 完整 Seurat 对象已保存:", paths$predictions$seurat_objects$xenium_annotated, "\n")

# 可选：保存 Flex 参考对象
if (params$save_intermediate && exists("flex_data.obj")) {
    saveRDS(flex_data.obj, paths$predictions$seurat_objects$flex_reference)
    cat("✅ Flex 参考对象已保存:", paths$predictions$seurat_objects$flex_reference, "\n")
}

# 生成统计报告
cat("\n", paste(rep("━", 60), collapse = ""), "\n")
cat("📊 生成统计报告...\n")
cat(paste(rep("━", 60), collapse = ""), "\n")

# 确保统计报告目录存在
stats_dir <- dirname(paths$reports$stats)
if (!dir.exists(stats_dir)) {
    dir.create(stats_dir, recursive = TRUE)
}

# 写入统计报告
sink(paths$reports$stats)

cat("预测结果统计报告\n")
cat("生成时间:", Sys.time(), "\n")
cat("总细胞数:", nrow(export_df), "\n")
cat(paste(rep("=", 60), collapse = ""), "\n\n")

# 1. 细胞类型分布
if ("predicted_cell_type" %in% colnames(export_df)) {
    cat("📈 1. 预测细胞类型分布\n")
    cat(paste(rep("-", 40), collapse = ""), "\n")
    type_counts <- sort(table(export_df$predicted_cell_type), decreasing = TRUE)
    print(type_counts)
}

# 2. 预测分数统计
if ("prediction_score" %in% colnames(export_df)) {
    valid_scores <- export_df$prediction_score[!is.na(export_df$prediction_score)]
    if (length(valid_scores) > 0) {
        cat("\n📈 2. 预测分数统计（有效分数: N=", length(valid_scores), "）\n", sep = "")
        cat(paste(rep("-", 40), collapse = ""), "\n")
        print(summary(valid_scores))
        
        cat("\n📈 3. 按细胞类型的预测分数统计\n")
        cat(paste(rep("-", 40), collapse = ""), "\n")
        score_by_type <- export_df %>%
            filter(!is.na(prediction_score)) %>%
            group_by(predicted_cell_type) %>%
            summarise(
                n_cells = n(),
                mean_score = mean(prediction_score, na.rm = TRUE),
                median_score = median(prediction_score, na.rm = TRUE),
                sd_score = sd(prediction_score, na.rm = TRUE),
                min_score = min(prediction_score, na.rm = TRUE),
                max_score = max(prediction_score, na.rm = TRUE),
                .groups = 'drop'
            ) %>%
            arrange(desc(mean_score))
        print(score_by_type)
    }
}

# 3. 一致性统计
if ("prediction_consistent" %in% colnames(export_df)) {
    cat("\n📈 4. 预测一致性统计\n")
    cat(paste(rep("-", 40), collapse = ""), "\n")
    consistent_count <- sum(export_df$prediction_consistent, na.rm = TRUE)
    total_valid <- sum(!is.na(export_df$prediction_consistent))
    if (total_valid > 0) {
        cat("  一致细胞数: ", consistent_count, "\n")
        cat("  不一致细胞数: ", total_valid - consistent_count, "\n")
        cat("  一致率: ", round(consistent_count / total_valid * 100, 2), "%\n", sep = "")
        
        cat("\n📈 5. 按细胞类型的一致性统计\n")
        cat(paste(rep("-", 40), collapse = ""), "\n")
        consistency_by_type <- export_df %>%
            filter(!is.na(prediction_consistent)) %>%
            group_by(predicted_cell_type) %>%
            summarise(
                n_cells = n(),
                consistent = sum(prediction_consistent, na.rm = TRUE),
                consistency_rate = consistent / n() * 100,
                .groups = 'drop'
            ) %>%
            arrange(desc(consistency_rate))
        print(consistency_by_type)
        
        # 不一致案例
        inconsistent_cases <- export_df %>%
            filter(!prediction_consistent & !is.na(prediction_consistent)) %>%
            select(cell_id, predicted_cell_type, max_score_type, prediction_score) %>%
            head(20)
        
        if (nrow(inconsistent_cases) > 0) {
            cat("\n📈 6. 不一致案例示例（前20个）\n")
            cat(paste(rep("-", 40), collapse = ""), "\n")
            print(inconsistent_cases)
        }
    }
}

sink()

cat("\n✅ 详细统计报告已保存:", paths$reports$stats, "\n")

In [ ]:
# ============================================================
# Cell 10: 论文图表补充 — 细胞类型组成 & 置信度热力图
#
# 图表列表（论文 Fig 2~5）
# A. scRNA Flex UMAP（细胞类型着色）
# B. Xenium 空间图（Seurat 预测细胞类型）
# C. 细胞类型比例堆叠条形图（scRNA vs Xenium Seurat）
# D. 各细胞类型平均预测置信度条形图
# E. Xenium 基因数量空间分布（QC 图）
# ============================================================

library(cowplot)
library(scales)

# ── A. scRNA UMAP（与 GNN 结果对比用）────────────────────────
cat("A. 绘制 scRNA UMAP...\n")
p_umap_scrna <- DimPlot(
    flex_data.obj,
    reduction = "umap",
    group.by  = "cell_type",
    label     = TRUE, repel = TRUE, pt.size = 0.3
) +
    ggtitle("Flex scRNA-seq — Cell Type Annotation (UMAP)") +
    theme(plot.title = element_text(hjust = 0.5, size = 12))
ggsave(file.path(paths$plots$subdirs$annotation, "scrna_umap_celltypes.pdf"),
       p_umap_scrna, width = 10, height = 8)
print(p_umap_scrna)
cat("  ✅ 保存: scrna_umap_celltypes.pdf\n")

# ── B. Xenium 空间图（Seurat 预测）────────────────────────────
if ("predicted.id" %in% colnames(xenium.obj@meta.data)) {
    cat("\nB. 绘制 Xenium 空间预测图（Seurat）...\n")
    p_spatial_seurat <- ImageDimPlot(
        xenium.obj,
        fov        = "fov",
        group.by   = "predicted.id",
        size       = 0.4,
        dark.background = FALSE
    ) +
        ggtitle("Seurat Label Transfer — Spatial Cell Types") +
        theme(plot.title = element_text(hjust = 0.5, size = 12),
              legend.text = element_text(size = 7))
    ggsave(file.path(paths$plots$subdirs$spatial, "xenium_spatial_seurat_predicted.pdf"),
           p_spatial_seurat, width = 12, height = 10)
    print(p_spatial_seurat)
    cat("  ✅ 保存: xenium_spatial_seurat_predicted.pdf\n")
}

# ── C. 细胞类型比例对比（堆叠条形图）────────────────────────
cat("\nC. 绘制细胞类型比例对比图...\n")

# 计算 scRNA 比例
scrna_types <- table(flex_data.obj$cell_type)
scrna_prop  <- data.frame(
    cell_type  = names(scrna_types),
    proportion = as.numeric(scrna_types) / sum(scrna_types),
    source     = "scRNA (Flex)"
)

# 计算 Xenium Seurat 比例
xen_prop_df <- NULL
if ("predicted.id" %in% colnames(xenium.obj@meta.data)) {
    xen_types <- table(xenium.obj$predicted.id)
    xen_prop_df <- data.frame(
        cell_type  = names(xen_types),
        proportion = as.numeric(xen_types) / sum(xen_types),
        source     = "Xenium (Seurat LT)"
    )
}

prop_combined <- rbind(scrna_prop, xen_prop_df)
n_types <- length(unique(prop_combined$cell_type))
type_colors <- setNames(
    hue_pal()(n_types),
    sort(unique(prop_combined$cell_type))
)

p_proportion <- ggplot(prop_combined,
       aes(x = source, y = proportion, fill = cell_type)) +
    geom_bar(stat = "identity", width = 0.6) +
    scale_fill_manual(values = type_colors) +
    scale_y_continuous(labels = percent_format()) +
    labs(title  = "Cell Type Proportion: scRNA vs Xenium (Seurat LT)",
         x = "", y = "Proportion", fill = "Cell Type") +
    theme_minimal(base_size = 11) +
    theme(plot.title   = element_text(hjust = 0.5),
          legend.position = "right",
          legend.text     = element_text(size = 8))
ggsave(file.path(paths$plots$subdirs$validation, "celltype_proportion_comparison.pdf"),
       p_proportion, width = 9, height = 6)
print(p_proportion)
cat("  ✅ 保存: celltype_proportion_comparison.pdf\n")

# ── D. 各细胞类型平均预测置信度条形图 ───────────────────────
cat("\nD. 绘制预测置信度条形图...\n")
if ("prediction.score.max" %in% colnames(xenium.obj@meta.data)) {
    score_df <- xenium.obj@meta.data %>%
        filter(!is.na(prediction.score.max) & !is.na(predicted.id)) %>%
        group_by(predicted.id) %>%
        summarise(
            mean_score   = mean(prediction.score.max),
            median_score = median(prediction.score.max),
            se_score     = sd(prediction.score.max) / sqrt(n()),
            n_cells      = n(),
            .groups = 'drop'
        ) %>%
        arrange(desc(mean_score))

    p_confidence <- ggplot(score_df,
           aes(x = reorder(predicted.id, mean_score),
               y = mean_score, fill = predicted.id)) +
        geom_col(show.legend = FALSE) +
        geom_errorbar(aes(ymin = mean_score - se_score,
                          ymax = mean_score + se_score),
                      width = 0.3, color = "gray40") +
        geom_text(aes(label = paste0("n=", n_cells)),
                  hjust = -0.15, size = 2.8) +
        scale_fill_viridis_d(option = "plasma") +
        scale_y_continuous(limits = c(0, 1.1), expand = c(0, 0)) +
        coord_flip() +
        labs(title = "Mean Prediction Score by Cell Type (Seurat LT)",
             x = "", y = "Mean Prediction Score") +
        theme_minimal(base_size = 11) +
        theme(plot.title = element_text(hjust = 0.5))

    ggsave(file.path(paths$plots$subdirs$validation, "prediction_confidence_by_type.pdf"),
           p_confidence, width = 9, height = 7)
    print(p_confidence)
    cat("  ✅ 保存: prediction_confidence_by_type.pdf\n")
}

# ── E. Xenium QC 空间图（基因数分布）──────────────────────────
cat("\nE. 绘制 Xenium QC 空间图...\n")
p_qc_spatial <- ImageFeaturePlot(
    xenium.obj,
    fov      = "fov",
    features = "nCount_Xenium_log",
    size     = 0.3
) +
    scale_fill_viridis_c(option = "magma", name = "log(nCount)") +
    ggtitle("Xenium Transcript Count per Cell (spatial)") +
    theme(plot.title = element_text(hjust = 0.5, size = 12))
ggsave(file.path(paths$plots$subdirs$quality_control, "xenium_qc_ncount_spatial.pdf"),
       p_qc_spatial, width = 10, height = 9)
print(p_qc_spatial)
cat("  ✅ 保存: xenium_qc_ncount_spatial.pdf\n")

cat("\n✅ 论文图表补充完成\n")
cat("  所有图表保存在:", paths$plots$dir, "\n")

In [ ]:
# ============================================================
# Cell 11: 导出 Seurat 标签转移结果 → Python 对比格式
#
# 输出：results/predictions/seurat_label_transfer_ovarian.csv
# 列：cell_id, x, y, predicted_id, prediction_score
#
# 此文件被 train_ovarian.ipynb Cell 9 加载，
# 与 GNN spot 预测做空间分布对比。
# ============================================================

cat("📤 导出 Seurat 标签转移结果（供 Python 对比）...\n")

if (!"predicted.id" %in% colnames(xenium.obj@meta.data)) {
    cat("❌ 未找到 predicted.id，请先运行 Cell 7 完成标签转移\n")
} else {
    # 提取坐标（优先用元数据里已有的）
    if ("x_centroid" %in% colnames(xenium.obj@meta.data)) {
        x_col <- xenium.obj@meta.data$x_centroid
        y_col <- xenium.obj@meta.data$y_centroid
    } else {
        coords_out <- GetTissueCoordinates(xenium.obj, scale = NULL)
        if ("cell" %in% colnames(coords_out)) {
            x_col <- coords_out$x[match(rownames(xenium.obj@meta.data), coords_out$cell)]
            y_col <- coords_out$y[match(rownames(xenium.obj@meta.data), coords_out$cell)]
        } else {
            x_col <- coords_out$x[match(rownames(xenium.obj@meta.data), rownames(coords_out))]
            y_col <- coords_out$y[match(rownames(xenium.obj@meta.data), rownames(coords_out))]
        }
    }

    # 置信度分数（优先用 prediction.score.max）
    if ("prediction.score.max" %in% colnames(xenium.obj@meta.data)) {
        score_col <- xenium.obj@meta.data$prediction.score.max
    } else if ("predicted.id_full.score" %in% colnames(xenium.obj@meta.data)) {
        score_col <- xenium.obj@meta.data$predicted.id_full.score
    } else {
        score_col <- NA_real_
    }

    lt_export <- data.frame(
        cell_id          = rownames(xenium.obj@meta.data),
        x                = x_col,
        y                = y_col,
        predicted_id     = xenium.obj@meta.data$predicted.id,
        prediction_score = score_col,
        stringsAsFactors = FALSE
    )

    # 过滤掉坐标缺失的行
    lt_export <- lt_export[!is.na(lt_export$x) & !is.na(lt_export$y), ]

    write.csv(lt_export, paths$seurat_lt_csv, row.names = FALSE)

    cat("✅ 导出完成 →", paths$seurat_lt_csv, "\n")
    cat("  细胞数 :", nrow(lt_export), "\n")
    cat("  列     :", paste(colnames(lt_export), collapse = ", "), "\n")
    cat("  坐标范围 x: [", round(min(lt_export$x, na.rm=TRUE),1), ",",
        round(max(lt_export$x, na.rm=TRUE),1), "]\n")
    cat("  坐标范围 y: [", round(min(lt_export$y, na.rm=TRUE),1), ",",
        round(max(lt_export$y, na.rm=TRUE),1), "]\n")
    cat("\n  预测类型分布:\n")
    print(sort(table(lt_export$predicted_id), decreasing = TRUE))
    cat("\n  平均置信度:",
        round(mean(lt_export$prediction_score, na.rm=TRUE), 3), "\n")
    cat("\n⚠️  此文件供 train_ovarian.ipynb Cell 9 使用，请勿移动路径。\n")
}


In [ ]:
# ============================================
# Cell 9.8: 导出完成摘要
# ============================================

cat("\n", paste(rep("━", 60), collapse = ""), "\n")
cat("📊 导出完成摘要\n")
cat(paste(rep("━", 60), collapse = ""), "\n")

cat("\n📁 输出文件:\n")
cat("  1. 基础预测结果:", paths$predictions$main, "\n")
if (file.exists(paths$predictions$full)) {
    cat("  2. 完整预测分数:", paths$predictions$full, "\n")
}
if (file.exists(paths$predictions$for_gnn)) {
    cat("  3. GNN 专用数据:", paths$predictions$for_gnn, "\n")
}
cat("  4. Seurat 对象:", paths$predictions$seurat_objects$xenium_annotated, "\n")
cat("  5. 统计报告:", paths$reports$stats, "\n")

cat("\n📈 数据统计:\n")
cat("  总细胞数:", nrow(export_df), "\n")

if (exists("valid_scores") && length(valid_scores) > 0) {
    cat("  有效分数细胞数:", length(valid_scores), 
        "(", round(length(valid_scores)/nrow(export_df)*100, 1), "%)\n", sep = "")
}

if (exists("consistent_count") && exists("total_valid") && total_valid > 0) {
    cat("  预测一致性: ", round(consistent_count / total_valid * 100, 1), "%\n", sep = "")
}

# 检查 GNN 数据质量
if (file.exists(paths$predictions$for_gnn)) {
    gnn_check <- read.csv(paths$predictions$for_gnn, nrows = 5)
    cat("\n🎯 GNN 数据质量检查:\n")
    cat("  文件存在 ✓\n")
    cat("  包含坐标列:", all(c("x_centroid", "y_centroid") %in% colnames(gnn_check)), "\n")
    cat("  包含 PCA 特征数:", sum(grepl("^PC_", colnames(gnn_check))), "\n")
}

cat("\n", paste(rep("━", 60), collapse = ""), "\n")
cat("✅ 所有导出任务完成
cat("  6. Seurat 对比导出:", paths$seurat_lt_csv, "\n")
cat("  7. 论文图表:", paths$plots$dir, "\n")\n")